In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from dateutil import parser
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from datetime import date

In [2]:
CSV      = "measurements_all_days.csv"
BAR_ON  = pd.Timestamp("2024-12-20")
BAR_OFF = pd.Timestamp("2025-01-17")
OUT_DIR  = Path("output_final")

In [3]:
df_raw = pd.read_csv(CSV)

In [4]:
dt_col = next(c for c in df_raw.columns if "date" in c.lower() or "time" in c.lower())
pm_col = next(c for c in df_raw.columns if "pm2" in c.lower() or "pm2.5" in c.lower())

In [5]:
def parse_dt(s: str):

    for fmt in ("%m/%d/%y %I:%M:%S %p", "%m/%d/%y %H:%M", "%m/%d/%y %H:%M:%S"):
        try:
            return pd.to_datetime(s, format=fmt)
        except Exception:
            pass
    try:
        return parser.parse(s, dayfirst=False)
    except Exception:
        return pd.NaT

In [6]:
df_raw[dt_col] = df_raw[dt_col].astype(str).apply(parse_dt)
df = (df_raw[[dt_col, pm_col]]
        .dropna()
        .rename(columns={dt_col: "datetime", pm_col: "PM25"})
        .sort_values("datetime")
        .set_index("datetime"))

In [7]:
if df.empty:
    raise SystemExit("✖ Данные не распарсились — проверьте исходный CSV")

In [8]:
US_HOLIDAYS = {
    date(2024, 12, 25),  # Christmas Day
    date(2025, 1, 1),    # New Year’s Day
    # add others as needed
}

In [9]:
def phase(ts: pd.Timestamp) -> str:
    h = ts.hour + ts.minute / 60

    # cooking always has priority
    if (9 <= h < 10) or (12 <= h < 14) or (20 <= h < 21):
        return "cooking"

    is_weekday = ts.weekday() < 5          # Monday–Friday
    holiday    = ts.date() in US_HOLIDAYS  # True if federal holiday

    if is_weekday and not holiday:
        if (8.5 <= h < 12) or (13 <= h < 17):
            return "facade"

    return "background"

In [10]:
def period(ts):
    if ts < BAR_ON:
        return "pre-barrier"
    elif ts < BAR_OFF:
        return "barrier-on"
    else:
        return "post-barrier"

In [11]:
df["phase"]  = df.index.map(phase)
df["period"] = df.index.map(period)
df["hour"]   = df.index.hour + df.index.minute/60

In [12]:
print("\nPhase counts:\n",   df.phase.value_counts())
print("\nPeriod counts:\n",  df.period.value_counts())


Phase counts:
 phase
background    4702
cooking       1168
facade        1061
Name: count, dtype: int64

Period counts:
 period
barrier-on      3886
post-barrier    2098
pre-barrier      947
Name: count, dtype: int64


In [13]:
df["baseline"] = np.nan
quad = make_pipeline(PolynomialFeatures(2), Ridge(alpha=1.0))

In [14]:
for p in df.period.unique():
    bg = df[(df.period == p) & (df.phase == "background")]
    if len(bg) < 10:
        df.loc[df.period == p, "baseline"] = bg.PM25.median() if len(bg) else df.PM25.median()
        print(f"• period {p}: baseline = median (not enough data)")
    else:
        quad.fit(bg[["hour"]], bg["PM25"])
        df.loc[df.period == p, "baseline"] = quad.predict(df.loc[df.period == p, ["hour"]])

In [15]:
df["delta"] = df["PM25"] - df["baseline"]

In [16]:
src = (df.groupby("phase")
         .agg(mean_PM25=("PM25","mean"),
              max_PM25=("PM25","max"),
              dose     =("delta", lambda x: x[x>0].sum())))
src["dose_pct"] = 100 * src.dose / src.dose.sum()

In [17]:
bg_period = (df[df.phase == "background"]
              .groupby("period")
              .agg(mean_PM25=("PM25","mean"),
                   p95_PM25=("PM25", lambda x: np.percentile(x, 95))))

In [18]:
print("\n=== Source contribution ===")
print(src.round(2))
print("\n=== Background per period ===")
print(bg_period.round(2))


=== Source contribution ===
            mean_PM25  max_PM25     dose  dose_pct
phase                                             
background       2.70     124.1  4727.89     45.36
cooking          4.25      66.5  2508.58     24.07
facade           5.16     161.7  3187.37     30.58

=== Background per period ===
              mean_PM25  p95_PM25
period                           
barrier-on         3.91     11.22
post-barrier       0.76      2.40
pre-barrier        1.86      3.70


In [19]:
bg_all = df[df.phase == "background"].copy()
X = bg_all[["hour"]].values
y = bg_all["PM25"].values

In [20]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_list = []

In [21]:
for train_idx, test_idx in kf.split(X):
    model = make_pipeline(PolynomialFeatures(2), Ridge(alpha=1.0))
    model.fit(X[train_idx], y[train_idx])
    y_pred = model.predict(X[test_idx])
    mae_list.append(mean_absolute_error(y[test_idx], y_pred))

In [22]:
mae_mean = np.mean(mae_list)
mae_std  = np.std(mae_list)
print(f"\n5-fold CV MAE = {mae_mean:.2f} ± {mae_std:.2f} µg/m³")


5-fold CV MAE = 2.30 ± 0.13 µg/m³


In [23]:
OUT_DIR.mkdir(exist_ok=True)

In [24]:
plt.figure(figsize=(8,4))
for p, g in df[df.phase=="background"].groupby("period"):
    g.groupby("hour").PM25.mean().plot(label=p)
plt.xlabel("Hour"); plt.ylabel("Mean PM₂.₅ (µg/m³)")
plt.title("Background PM₂.₅ — three periods"); plt.legend()
plt.tight_layout(); plt.savefig(OUT_DIR/"baseline_periods.png", dpi=300); plt.close()

In [25]:
src.loc[["background","facade","cooking"]].dose_pct.plot(
        kind="bar", figsize=(4,4), ylabel="Dose Share (%)",
        title="Source Contribution to Daily PM₂.₅ Dose")
plt.tight_layout(); plt.savefig(OUT_DIR/"dose_share.png", dpi=300); plt.close()

In [26]:
colors = {"pre-barrier":"tab:blue", "barrier-on":"tab:orange", "post-barrier":"tab:green"}
plt.figure(figsize=(10,3))
for p, c in colors.items():
    df[df.period == p].PM25.plot(lw=0.6, color=c, label=p)
plt.ylabel("PM₂.₅ (µg/m³)"); plt.title("PM₂.₅ Time Series (period colour)")
plt.legend(); plt.tight_layout()
plt.savefig(OUT_DIR/"timeline_colored.png", dpi=300); plt.close()

In [27]:
print(f"\nPNG files saved to {OUT_DIR.resolve()}")


PNG files saved to /Users/rostyslavsipakov/Documents/GitHub/QuadraticPolynomialsPyDA/notebooks/knuba_cook_facade/output_final
